# D6-信令数据洞察-驻留识别

流水线能跑了，我们开始回答业务问题：一个人什么时候停留在哪里？停留识别的参数第一次需要一个配置文件

## 从信令到停留点
看懂一个人一天的信令轨迹；解释“人没动、基站在变”的乒乓切换；写出停留识别算法，把 50 多万条信令压缩成约 2 万个停留点；用敏感性分析选参数；把停留表 dws_stay 写回库里。

In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2 
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT   # 配色常量

viz.apply_theme()                          # 字体、配色、坐标轴样式：一行
con = sqlite3.connect(config.DB_PATH)      # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                  # 分析日期（口径常量）
NB = "D06_驻留识别"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"   # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
hot = pd.read_sql("SELECT * FROM dim_hotspot", con).iloc[0]
DAY_START = pd.Timestamp(DAY)
def mask_id(imsi):
    """展示用户时：只显示IMSI尾号，不在图和表里暴露完整ID"""
    return "尾号" + str(imsi)[-4:]

In [ ]:
dwd = pd.read_sql("SELECT imsi, ts, lac_cell FROM dwd_signal",
                con, parse_dates=['ts'])
dim = pd.read_sql("SELECT lac_cell, site_id, lng, lat FROM dim_cell", con)
sig = (dwd.merge(dim, on='lac_cell', how='inner')).sort_values(
    ['imsi','ts'],ignore_index=True
)
sig['到西塔(米)'] = geo.haversine(sig['lat'], sig['lng'], hot['lat'], hot['lng'])
n_users = sig['imsi'].nunique()

## 需求：一个人的一天
挑一个典型用户-凌晨不在西塔，晚上到过西塔，把这一天的信令画出来：横轴是时间，纵轴是到西塔的距离

In [ ]:
h = sig["ts"].dt.hour
night_far = sig[(h == 3) & (sig["到西塔(米)"] > 3000)]["imsi"].unique()     
evening_in = sig[(h.between(19, 20)) &  # 19–20 点在西塔 500 米内
            (sig["到西塔(米)"] <= 500)]["imsi"].unique()  
n_rec = sig.groupby("imsi").size()
candidates = n_rec[n_rec.index.isin(night_far) &
                n_rec.index.isin(evening_in) & n_rec.between(60, 120)]
demo_id = candidates.index[0]   # 按 IMSI 排序取第一个，保证每次选到同一个人
demo = sig[sig["imsi"] == demo_id]
print(f"候选 {len(candidates)} 人；示例用户 {mask_id(demo_id)}，当天 {len(demo)} 条信令")

In [ ]:
hour = (demo["ts"] - DAY_START).dt.total_seconds() / 3600          # 时间换算成“几点”（小数）
km = demo["到西塔(米)"] / 1000
arrive = demo[(demo["ts"].dt.hour >= 12) &  # 到达西塔的时间
            (demo["到西塔(米)"] <= 500)]["ts"].min()
night_km = km[demo["ts"].dt.hour < 6].median()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axhspan(0, 0.5, color=MAIN_LIGHT, lw=0, zorder=0)                     # 西塔 500 米范围
ax.text(0.2, 0.25, "西塔 500 米内", va="center", color=MAIN, fontsize=9.5)
ax.plot(hour, km, color=GRAY, lw=1, zorder=1)
ax.scatter(hour, km, s=16, color=MAIN, zorder=2)
ax.annotate(f"{arrive:%H:%M} 到达西塔", (hour[demo["ts"] == arrive].iloc[0], 0.5), xytext=(25, 70),
            textcoords="offset points", ha="left", fontsize=10, color=INK, arrowprops=dict(arrowstyle="->", color=FAINT))
ax.set_xticks(range(0, 25, 3), [f"{x} 点" for x in range(0, 25, 3)])
ax.set_xlim(0, 24)
ax.set_ylim(0, km.max() * 1.15)
ax.set_ylabel("离西塔的距离（公里）", color=MUTED)

title = f"{mask_id(demo_id)} 的一天：凌晨在距西塔约 {night_km:.1f} 公里处，{arrive:%H:%M} 到达西塔"
viz.finish(fig, title,
           f"2018-10-03 该用户每条信令所连站点到西塔中心的距离；共 {len(demo)} 条；口径：dwd_signal × dim_cell",
           SOURCE, "D6_01_一个人的一天.png")
plt.show()

## 信令乒乓切换：人没动，基站在变
需求：相邻的两个记录之间，有多少换了站点，换站点时两站相距多远？

In [ ]:
g = sig.groupby('imsi')
prev_site = g['site_id'].shift()
switch = prev_site.notna() & (sig['site_id'] != prev_site) # 和上一条相比换了站点
jump_m = geo.haversine(sig['lat'], sig['lng'], g['lat'].shift(), g['lng'].shift())
switch_m = pd.Series(jump_m[switch.to_numpy()])

In [ ]:
# A -> B -> A的判定：当前这条换了站点switch，而且他的下一条又回到上一条的站点
# 并且上一条到下一条之间不超过10分钟，shift(-1)取下一条，shift()取上一条
aba = switch & (g['site_id'].shift(-1) == prev_site) & \
    ((g['ts'].shift(-1) - g['ts'].shift()).dt.total_seconds() <= 600)
print(f"相邻记录换了站点：{switch.mean():.0%}")
print(f"换站点时两站地距离中位数：{switch_m.median():.1f}米, \
{(switch_m < 1000).mean():.1%}在一公里以内")
print(f"10分钟内又切换为原站点(A->B-A):占全部切换到{aba.sum() / switch.sum():.1%}")

In [ ]:
# ── 出图 D6_02：直方图 ──────────────────────────────────────────────────
# 为什么用直方图而不是 ECDF：这里要让读者看见**形状**——绝大多数切换挤在最左边一两公里内，
# 右边拖一条长尾。形状本身就是"人没动、基站在变"的证据。
# （D4_03 要回答"多少人在 500 米内"这种累计问题，那里才用 ECDF。两者的分工就在这。）
# 规范落在哪几行：
#   · 纵轴用**占比**不用计数 —— 换一天数据、样本量变了，图的高度还能横着比
#   · clip(upper=4999) 把 5 公里以上并进最后一组，横轴标签写明"最后一组含 5 公里以上" ——
#     长尾不截断的话，右边 90% 的画布全是空白
#   · 1 公里以内上主色、以外灰色 —— 把"要强调的那部分"直接涂出来，不靠读者自己判断
# 标题里的数字：switch.mean()（换站点的比例）、p50（切换距离的中位数）。
bins = np.arange(0, 5001, 250)
counts, _ = np.histogram(switch_m.clip(upper=4999), bins=bins)
share = counts / counts.sum()
p50 = switch_m.median()

fig, ax = plt.subplots(figsize=(10, 5.6))
colors = [MAIN if b < 1000 else GRAY for b in bins[:-1]]
ax.bar(bins[:-1] / 1000, share, width=0.23, align="edge", color=colors)
ax.axvline(p50 / 1000, color=INK, lw=1, ls=(0, (4, 3)))
ax.text(p50 / 1000, share.max() * 1.02, f"  中位数 {p50:,.0f} 米", color=INK, fontsize=10)
ax.set_xticks(range(0, 6), [f"{x} 公里" if x else "0" for x in range(0, 6)])
ax.set_xlabel("切换前后两个站点的距离（最后一组含 5 公里以上）", color=MUTED)
ax.yaxis.set_major_formatter("{x:.0%}")
ax.set_ylim(0, share.max() * 1.15)

title = f"{switch.mean():.0%} 的相邻信令换了站点，一半的切换距离不到 {p50:,.0f} 米"
viz.finish(fig, title,
           "站点切换距离的分布，每 250 米一组；蓝色 = 1 公里以内，多为人没动时相邻基站之间的切换；口径：dwd_signal × dim_cell",
           SOURCE, "D6_02_站点切换距离.png")
plt.show()

## 停留如何识别？
- 思路：把每个人按时间拍好点信令从头往后扫=
    - 第一条记录开一个段，记录段的质心。段内所有点的平均位置
    - 下一条记录：离质心不超过R米，我把你并入这个段，并且根据两个点更新下质心；超过R米，就结束这一段，从这个记录开始开一个新的段
    - 扫完之后，持续时间不少于T分钟的段叫停留，其余的是经过
    
**半径R用来吸收兵乓切换，时长T用来区分停下来过和路过**，先用R=500米，T = 30分钟 

In [ ]:
import math
import time
def dist_m(lat1, lng1, lat2, lng2):
    """两点距离（米），标量版：循环里一次只算一对点，用 math 比用 numpy 快。"""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2
    return 2 * 6_371_000 * math.asin(math.sqrt(a))

def find_segments(df, radius_m = 500):
    """把每个人按照时间排好的信令切成段：离当前段质心不超过radius_m就并入，否则开新段"""
    imsi = df['imsi'].to_numpy()
    t = (df['ts'] - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
    lat, lng = df['lat'].to_numpy(), df['lng'].to_numpy()
    rows, i, n = [], 0, len(df)
    while i < n:
        sum_lat, sum_lng, k, j = lat[i], lng[i], 1, i
        while j + 1 < n and imsi[j + 1] == imsi[i]:
            if dist_m(sum_lat / k, sum_lng / k, lat[j+1], lng[j+1]) > radius_m:
                break
            j += 1
            sum_lat, sum_lng, k = sum_lat + lat[j], sum_lng + lng[j], k + 1
        rows.append((imsi[i], t[i], t[j], sum_lat / k, sum_lng / k, k))
        i = j + 1
    seg = pd.DataFrame(rows,
            columns=['imsi', "t_start", 't_end', "lat", 'lng','n_records'])
    seg['duration_min'] = (seg['t_end'] - seg['t_start']) / 60
    return seg

In [ ]:
t0 = time.perf_counter()
seg = find_segments(sig, radius_m=500)
stay = seg[seg["duration_min"] >= 30].reset_index(drop=True)
print(f"{len(sig):,} 条信令 → {len(seg):,} 个段 → {len(stay):,} 个停留（≥ 30 分钟），用时 {time.perf_counter() - t0:.1f} 秒")
print(f"有停留的用户：{stay['imsi'].nunique():,} 人（{stay['imsi'].nunique() / n_users:.1%}），人均 {len(stay) / n_users:.2f} 个停留")


```text
sig.head().to_dict() 数据内容如下：一些信令数据，数据差不多有几十万条，有很多用户，这里仅仅拿到前10个
{'imsi': {0: '460000251922',
  1: '460000251922',
  2: '460000251922',
  3: '460000251922',
  4: '460000251922'},
 'ts': {0: Timestamp('2018-10-03 00:50:19.870000'),
  1: Timestamp('2018-10-03 02:35:10.770000'),
  2: Timestamp('2018-10-03 04:07:03.069000'),
  3: Timestamp('2018-10-03 04:57:00.055000'),
  4: Timestamp('2018-10-03 05:46:57.963000')},
 'lac_cell': {0: '16794-235003671',
  1: '16794-235003671',
  2: '16794-235003671',
  3: '16794-235003671',
  4: '16794-235003671'},
 'site_id': {0: 'S11408', 1: 'S11408', 2: 'S11408', 3: 'S11408', 4: 'S11408'},
 'lng': {0: 123.3886108,
  1: 123.3886108,
  2: 123.3886108,
  3: 123.3886108,
  4: 123.3886108},
 'lat': {0: 41.79774857,
  1: 41.79774857,
  2: 41.79774857,
  3: 41.79774857,
  4: 41.79774857},
 '到西塔(米)': {0: 2088.601918510513,
  1: 2088.601918510513,
  2: 2088.601918510513,
  3: 2088.601918510513,
  4: 2088.601918510513}}
我们当前在想如何计算停留， 停留如何识别？
思路：把每个人按时间拍好点信令从头往后扫=
第一条记录开一个段，记录段的质心。段内所有点的平均位置
下一条记录：离质心不超过R米，我把你并入这个段，并且根据两个点更新下质心；超过R米，就结束这一段，从这个记录开始开一个新的段
扫完之后，持续时间不少于T分钟的段叫停留，其余的是经过
半径R用来吸收兵乓切换，时长T用来区分停下来过和路过，先用R=500米，T = 30分钟

我是这样实现的，但是感觉有一些性能分析，而且代码很不好理解
```
```python
import math
import time
def dist_m(lat1, lng1, lat2, lng2):
    """两点距离（米），标量版：循环里一次只算一对点，用 math 比用 numpy 快。"""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2
    return 2 * 6_371_000 * math.asin(math.sqrt(a))

def find_segments(df, radius_m = 500):
    """把每个人按照时间排好的信令切成段：离当前段质心不超过radius_m就并入，否则开新段"""
    imsi = df['imsi'].to_numpy()
    t = (df['ts'] - pd.Timestamp("1970-01-01")).dt.total_seconds().to_numpy()
    lat, lng = df['lat'].to_numpy(), df['lng'].to_numpy()
    rows, i, n = [], 0, len(df)
    while i < n:
        sum_lat, sum_lng, k, j = lat[i], lng[i], 1, i
        while j + 1 < n and imsi[j + 1] == imsi[i]:
            if dist_m(sum_lat / k, sum_lng / k, lat[j+1], lng[j+1]) > radius_m:
                break
            j += 1
            sum_lat, sum_lng, k = sum_lat + lat[j], sum_lng + lng[j], k + 1
        rows.append((imsi[i], t[i], t[j], sum_lat / k, sum_lng / k, k))
        i = j + 1
    seg = pd.DataFrame(rows,
            columns=['imsi', "t_start", 't_end', "lat", 'lng','n_records'])
    seg['duration_min'] = (seg['t_end'] - seg['t_start']) / 60
    return seg
```
```text
关于距离计算有写好的轮子：
Help on function haversine in module signal_insight.geo:
```

### 更好读的版本

In [ ]:
"""停留识别：把每个人按时间排好的信令切成段，时长够的段叫停留。"""
import math
import numpy as np
import pandas as pd
__all__ = ["find_segments"]
EARTH_R = 6_371_000

def _dist_m(lat1, lng1, lat2, lng2):
    """标量版 haversine（米）。逐点循环里用 math 比 numpy 版快一个数量级。"""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2
    return 2 * EARTH_R * math.asin(math.sqrt(a))


def _label_segments(new_user, lat, lng, w, radius_m):
    """顺序扫描，给每个点打段号。算法里唯一必须逐点做的部分。
    new_user[i]: 第 i 个点是不是某人的第一个点（第 0 个必须是 True）
    w[i]:        第 i 个点代表几条原始记录（用于加权算质心）
    """
    labels = np.empty(len(lat), dtype=np.int64)
    seg, s_lat, s_lng, s_w = -1, 0.0, 0.0, 0.0
    for i in range(len(lat)):
        if new_user[i] or _dist_m(s_lat / s_w, s_lng / s_w, lat[i], lng[i]) > radius_m:
            seg, s_lat, s_lng, s_w = seg + 1, 0.0, 0.0, 0.0      # 换人或走远了：开新段
        s_lat += lat[i] * w[i]
        s_lng += lng[i] * w[i]
        s_w += w[i]
        labels[i] = seg
    return labels


def find_segments(df: pd.DataFrame, radius_m: float = 500, min_stay_min: float = 30) -> pd.DataFrame:
    """信令 → 段表。离当前段质心不超过 radius_m 就并入，否则开新段；
    时长不少于 min_stay_min 分钟的段 is_stay=True，其余是经过。
    df 至少要有 imsi、ts、lat、lng。
    """
    df = df.sort_values(["imsi", "ts"], ignore_index=True)
    imsi, lat, lng = (df[c].to_numpy() for c in ("imsi", "lat", "lng"))

    # 1. 连续落在同一坐标的记录必然同段，先压成一行，循环次数大幅减少
    new_user = np.r_[True, imsi[1:] != imsi[:-1]]
    moved = new_user | np.r_[True, (lat[1:] != lat[:-1]) | (lng[1:] != lng[:-1])]
    starts = np.flatnonzero(moved)
    counts = np.diff(np.r_[starts, len(df)])

    # 2. 在压缩后的点上顺序扫描打段号，再展开回原始记录
    run_seg = _label_segments(new_user[starts], 
            lat[starts], lng[starts], counts.astype(float), radius_m)
    df["seg"] = np.repeat(run_seg, counts)

    # 3. 按段汇总：质心就是段内原始记录的平均位置
    seg = df.groupby("seg").agg(
        imsi=("imsi", "first"),
        t_start=("ts", "first"),
        t_end=("ts", "last"),
        lat=("lat", "mean"),
        lng=("lng", "mean"),
        n_records=("ts", "size"),
    ).reset_index(drop=True)
    seg["duration_min"] = (seg["t_end"] - seg["t_start"]).dt.total_seconds() / 60
    seg["is_stay"] = seg["duration_min"] >= min_stay_min
    return seg

In [ ]:
t0 = time.perf_counter()
seg = find_segments(sig, radius_m=500)
stay = seg[seg["duration_min"] >= 30].reset_index(drop=True)
print(f"{len(sig):,} 条信令 → {len(seg):,} 个段 → {len(stay):,} 个停留（≥ 30 分钟），用时 {time.perf_counter() - t0:.1f} 秒")
print(f"有停留的用户：{stay['imsi'].nunique():,} 人（{stay['imsi'].nunique() / n_users:.1%}），人均 {len(stay) / n_users:.2f} 个停留")

## 参数怎么订：敏感性分析
需求：R=500米，T=30分钟这种决定书拍脑袋决定的吗？我们需要换几个值试一试，看结果变化大不大

::: IMPORTANT 重点
敏感性分析是企业里面定参数的标准做法，结果对某个参数不敏感，说明选哪个都差不多，用行业值即可。如果参数对这个值很敏感，就需要在报告里面写明参数，并说明你选择的理由。
:::

In [ ]:
# 这一格在做什么：敏感性分析——把半径和最短时长各试几个值，看结果变多少。
#
# 为什么必须做：500 米、30 分钟这两个数不是算出来的，是**选**出来的。
# 答辩时客户一定会问"为什么是 500 米"。能接住这个问题的回答只有一种：
# "我试过 300 到 1000 米，人均停留点只从 3.4 变到 3.2，结论对这个参数不敏感。"
#
# 讲台提示：企业里改参数要能说出依据，这是本课程教的"可辩护的参数选择"。
# 先让学生猜"半径放宽一倍，停留点会多还是少、变化大不大"，再跑。
grid = []
for r in [200, 300, 500, 800, 1000]:
    s = find_segments(sig, radius_m=r)
    for t_min in [5, 15, 30, 45, 60]:
        grid.append({"半径(米)": r, "最短时长(分)": t_min,
                     "人均停留点": (s["duration_min"] >= t_min).sum() / n_users})
grid = pd.DataFrame(grid)
grid.to_sql("ads_stay_sensitivity", con, if_exists="replace", index=False)

In [ ]:
# ── 出图：停留参数敏感性 ────────────────────────────────────────────
# 规范第 4 节"参数影响 → 多条折线，选用的参数加粗并圈出"：
# 横轴半径，每条线一个最短时长；选用的 30 分钟用主色加粗，其余灰色打底；
# 线尾直接写时长代替图例；选用点画圈。标题、注释里的数字全部由 grid 算出。
import matplotlib.pyplot as plt

R_SEL, T_SEL = 500, 30                                   # 选用的参数

tab = grid.pivot(index="半径(米)", columns="最短时长(分)", values="人均停留点")
radii, mins = tab.index.to_numpy(), tab.columns.to_numpy()
r_lo, r_hi = radii.min(), radii.max()

sel = tab[T_SEL]
base, lo, hi = sel[R_SEL], sel.min(), sel.max()
t_short, t_long = tab[mins.min()], tab[mins.max()]     # 两端的时长：看半径效应的方向
chg_short = t_short[r_hi] / t_short[r_lo] - 1           # 时长门槛低：半径放大 → 变少
chg_long = t_long[r_hi] / t_long[r_lo] - 1              # 时长门槛高：半径放大 → 变多
pct = lambda v, d=0: f"{v:+.{d}%}".replace("-", "−")    # 真负号；规范第 6 节：接近的数保留一位

fig, ax = plt.subplots(figsize=(10, 5.6))

# 线尾标签：末端值太近时（45、60 分钟只差 0.23）自下而上推开，保证最小间距
GAP = 0.4                         # 标签最小间距（纵轴单位）
ends = tab.loc[r_hi].sort_values()
label_y, prev = {}, -float("inf")
for t, y in ends.items():
    label_y[t] = prev = max(y, prev + GAP)

for t in mins:
    chosen = t == T_SEL
    ax.plot(radii, tab[t], color=MAIN if chosen else GRAY, lw=2.8 if chosen else 1.5,
            marker="o", ms=5 if chosen else 3.5, zorder=4 if chosen else 2)
    ax.annotate(f"{t} 分钟", xy=(r_hi, tab[t][r_hi]), xytext=(r_hi + 45, label_y[t]),
                va="center", fontsize=10, annotation_clip=False,
                color=MAIN if chosen else MUTED, fontweight="bold" if chosen else "normal")
ax.annotate("最短时长", xy=(r_hi + 45, max(label_y.values()) + 0.45),
            va="center", fontsize=9, color=FAINT, annotation_clip=False)

# 选用口径下的波动范围：浅主色横带
ax.axhspan(lo, hi, color=MAIN_LIGHT, zorder=1, lw=0)

# 圈出选用点
ax.scatter([R_SEL], [base], s=260, facecolors="none", edgecolors=MAIN, lw=1.6, zorder=5)
ax.annotate(f"选用：{R_SEL:,} 米、{T_SEL} 分钟，人均 {base:.2f} 个\n"
            f"{r_lo:,}–{r_hi:,} 米都落在浅蓝带 {lo:.2f}–{hi:.2f} 内\n"
            f"相对选用值 {pct(lo / base - 1, 1)}～{pct(hi / base - 1, 1)}",
            xy=(R_SEL, base), xytext=(R_SEL + 130, 0.85), fontsize=9.5, color=INK,
            va="center", linespacing=1.5,
            arrowprops=dict(arrowstyle="-", color=MAIN, lw=1, shrinkA=2, shrinkB=9))

# 两种方向相反的效应：解释为什么 30 分钟附近对半径不敏感
ax.annotate(f"门槛低时，半径放大把乒乓碎段并起来，停留点变少（{pct(chg_short)}）",
            xy=(r_lo, t_short[r_lo]), xytext=(r_lo + 15, t_short[r_lo] + 0.45),
            fontsize=9, color=MUTED, va="bottom")
ax.annotate(f"门槛高时，半径放大把段拼长，够时长的段变多（{pct(chg_long)}）",
            xy=(r_lo, t_long[r_lo]), xytext=(r_lo + 15, t_long[r_lo] - 0.45),
            fontsize=9, color=MUTED, va="top")

ax.set_xticks(radii, [f"{r:,}" for r in radii])
ax.set_xlim(r_lo - 40, r_hi + 40)
ax.set_ylim(0, tab.to_numpy().max() + 1)
ax.set_xlabel("半径（米）")
ax.set_ylabel("人均停留点（个）")

title = f"{T_SEL} 分钟口径下，半径 {r_lo:,}–{r_hi:,} 米人均停留点只差 {hi - lo:.2f} 个"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"{sig['ts'].min():%m-%d}–{sig['ts'].max():%m-%d}，{n_users:,} 人；"
            f"离段质心不超过半径的连续记录合为一段，时长不少于最短时长计为停留\n"
            f"人均停留点 = 停留段数 ÷ 总人数；只测了图中 {len(radii)}×{len(mins)} 个组合；纵轴从 0 开始")

viz.finish(fig, title, subtitle, SOURCE, "D05_04_停留参数敏感性.png")
plt.show()

In [ ]:
# ── 出图：停留参数敏感性（重设计）────────────────────────────────────
# 上图 A：只改一个参数（另一个固定在选用值），看结果能变多大 —— 回答"为什么是 500 米"
# 下图 B：全部 25 个组合的热力矩阵（规范第 4 节"矩阵 → 热力矩阵，格子里写数字"），
#         右侧一列写"半径从最小放到最大"的变化，解释 30 分钟为什么不敏感
# 标题、注释里的数字全部由 grid 计算
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle

R_SEL, T_SEL = 500, 30                                     # 选用的参数

# 行=时长，列=半径。先取两位小数：图上写的就是这些数，标题里的百分比也从它们算，读者用图上的数能算回标题
tab = grid.pivot(index="最短时长(分)", columns="半径(米)", values="人均停留点").round(2)
mins, radii = tab.index.to_numpy(), tab.columns.to_numpy()
base = tab.loc[T_SEL, R_SEL]
by_r = tab.loc[T_SEL]                                      # 时长固定 30 分钟，只改半径
by_t = tab[R_SEL]                                          # 半径固定 500 米，只改时长
swing = lambda s: s.max() / s.min() - 1                    # "差多少" = 最高比最低多多少
r_swing, t_swing = swing(by_r), swing(by_t)
r_chg = tab[radii.max()] / tab[radii.min()] - 1            # 每个时长下：半径从最小放到最大的变化
pct = lambda v, fmt="+.0%": format(v, fmt).replace("-", "−")
fmt_r = lambda r: f"{r:,}"

# 用 constrained 布局：viz.finish 认得它，会把标题区、来源行的留白交给它统一排。
# 不要在 gridspec_kw 里写 hspace/wspace —— 那会让 tight_layout 判定"不兼容"、整张图排版失效。
fig, axd = plt.subplot_mosaic([["A", "A"], ["B", "N"]], figsize=(10, 5.6), layout="constrained",
                              height_ratios=[1, 1.75], width_ratios=[2.3, 1])
fig.get_layout_engine().set(h_pad=0.12, hspace=0.06, wspace=0.0)   # 两块之间多留一点
axA, axB, axN = axd["A"], axd["B"], axd["N"]

def heading(ax, text):
    """分图小标题：用 axes 标题而不是自由摆放的 text，布局引擎才会给它留位置、不和别的字压在一起。"""
    ax.set_title(text, loc="left", fontsize=11.5, fontweight="bold", color=INK, pad=12)

# ── A：只改一个参数 ──────────────────────────────────────────────
SW_X = tab.to_numpy().max() + 0.2                       # "差 X%" 那一列的横坐标
WHITE_BOX = dict(fc="white", ec="none", pad=1.5)          # 文字压在虚线上时垫白底
rows = [  # (y, 序列, 每个点的名字, 选用值对应的键)
    (1, by_r, {r: f"{fmt_r(r)} 米" for r in radii}, R_SEL),
    (0, by_t, {t: f"{t} 分钟" for t in mins}, T_SEL),
]
for y, s, names, key in rows:
    axA.plot([s.min(), s.max()], [y, y], color=MAIN_LIGHT, lw=11, solid_capstyle="round", zorder=1)
    others = s.drop(key)
    axA.scatter(others, [y] * len(others), s=34, color=GRAY_DARK, edgecolor="white", lw=0.8, zorder=3)
    axA.scatter([s[key]], [y], s=90, color=MAIN, edgecolor="white", lw=1.2, zorder=4)
    axA.text(SW_X, y, f"差 {pct(swing(s), '.0%')}", va="center", ha="left", fontsize=11,
             fontweight="bold", color=MAIN if y == 1 else INK)

# 半径那一行点挤在一起：只标两端
names_r = rows[0][2]
axA.text(by_r.min() - 0.12, 1, f"{names_r[by_r.idxmin()]} {by_r.min():.2f}",
         ha="right", va="center", fontsize=9, color=MUTED)
axA.text(by_r.max() + 0.12, 1, f"{by_r.max():.2f} {names_r[by_r.idxmax()]}",
         ha="left", va="center", fontsize=9, color=MUTED)
# 时长那一行：上面写时长、下面写数值；单位只写在最右一个上
for t, v in by_t.items():
    sel = t == T_SEL
    axA.text(v, 0.3, f"{t} 分钟" if t == mins.min() else f"{t}", ha="center", va="bottom",
             fontsize=9, color=MAIN if sel else MUTED, fontweight="bold" if sel else "normal",
             bbox=WHITE_BOX if sel else None, zorder=5)
    axA.text(v, -0.3, f"{v:.2f}", ha="center", va="top", fontsize=9, color=MAIN if sel else MUTED,
             bbox=WHITE_BOX if sel else None, zorder=5)

axA.axvline(base, color=MAIN, lw=1, ls=(0, (3, 3)), zorder=0)
axA.text(base, 1.42, f"选用 {fmt_r(R_SEL)} 米、{T_SEL} 分钟：{base:.2f}", ha="center", va="bottom",
         fontsize=9, color=MAIN, bbox=WHITE_BOX)
axA.text(SW_X, 1.42, "最高比最低", ha="left", va="bottom", fontsize=9, color=FAINT)

axA.set_yticks([1, 0], [f"只改半径\n时长定 {T_SEL} 分钟", f"只改最短时长\n半径定 {fmt_r(R_SEL)} 米"],
               fontsize=10)
axA.tick_params(axis="y", length=0)
axA.set_ylim(-0.75, 1.75)
axA.set_xlim(2, SW_X + 0.55)
axA.set_xlabel("人均停留点（个）")
axA.grid(axis="x", color="#EEF0F3"); axA.grid(axis="y", visible=False)
for side in ("left", "top", "right"):
    axA.spines[side].set_visible(False)
heading(axA, "A　只改一个参数：半径几乎不影响结果，最短时长影响很大")

# ── B：全部 25 个组合 ────────────────────────────────────────────
cmap = LinearSegmentedColormap.from_list("main_seq", ["#F4F8FC", MAIN_MID, MAIN])   # 单色渐变
norm = Normalize(tab.to_numpy().min(), tab.to_numpy().max())
axB.imshow(tab.to_numpy(), cmap=cmap, norm=norm, aspect="auto")
for i, t in enumerate(mins):
    for j, r in enumerate(radii):
        v = tab.loc[t, r]
        axB.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10,
                 color="white" if norm(v) > 0.55 else INK,
                 fontweight="bold" if (t, r) == (T_SEL, R_SEL) else "normal")
i0, j0 = list(mins).index(T_SEL), list(radii).index(R_SEL)
axB.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 1, 1, fill=False, ec=INK, lw=2.2))   # 圈出选用格
axB.add_patch(Rectangle((-0.5, i0 - 0.5), len(radii), 1, fill=False, ec=MAIN, lw=1, ls=(0, (3, 2))))

axB.set_xticks(range(len(radii)), [fmt_r(r) for r in radii])
axB.set_yticks(range(len(mins)), [f"{t} 分钟" for t in mins])
for lab, val in zip(axB.get_xticklabels(), radii):
    if val == R_SEL: lab.set_color(MAIN); lab.set_fontweight("bold")
for lab, val in zip(axB.get_yticklabels(), mins):
    if val == T_SEL: lab.set_color(MAIN); lab.set_fontweight("bold")
axB.set_xlabel("半径（米）")
axB.set_ylabel("最短时长")
axB.tick_params(length=0)
axB.grid(False)
for sp in axB.spines.values():
    sp.set_visible(False)
heading(axB, "B　全部 25 个组合：半径的影响方向随最短时长翻转，30 分钟附近正好抵消")

# 右侧注释栏：与热力矩阵共用行
axN.set_ylim(axB.get_ylim()); axN.set_xlim(0, 1); axN.axis("off")
axN.text(0.04, len(mins) - 0.5 + 0.12, f"半径 {fmt_r(radii.min())}→{fmt_r(radii.max())} 米\n停留点的变化",
         fontsize=9, color=FAINT, va="top", linespacing=1.4)
for i, t in enumerate(mins):
    sel = t == T_SEL
    axN.text(0.04, i, pct(r_chg[t]), va="center", fontsize=10.5,
             color=MAIN if sel else INK, fontweight="bold" if sel else "normal")

def bracket(i_from, i_to, text, color=MUTED):
    x = 0.33
    axN.plot([x, x + 0.04, x + 0.04, x], [i_from - 0.3, i_from - 0.3, i_to + 0.3, i_to + 0.3],
             color=GRAY, lw=1)
    axN.text(x + 0.1, (i_from + i_to) / 2, text, va="center", fontsize=9, color=color, linespacing=1.45)

neg = [i for i, t in enumerate(mins) if r_chg[t] < 0 and t < T_SEL]
posi = [i for i, t in enumerate(mins) if r_chg[t] > 0]
if neg:  bracket(min(neg), max(neg), "门槛低：碎段被合并\n半径越大，停留点越少")
axN.text(0.43, i0, "两种效应接近抵消", va="center", fontsize=9, color=MAIN)
if posi: bracket(min(posi), max(posi), "门槛高：段被拼长\n半径越大，停留点越多")

# ── 标题、副标题、来源 ───────────────────────────────────────────
title = (f"半径 {fmt_r(radii.min())}–{fmt_r(radii.max())} 米只差 {pct(r_swing, '.0%')}，"
         f"最短时长 {mins.min()}–{mins.max()} 分钟差 {pct(t_swing, '.0%')}")
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
d0, d1 = sig["ts"].min(), sig["ts"].max()
period = f"{d0:%m-%d}" if d0.date() == d1.date() else f"{d0:%m-%d}–{d1:%m-%d}"
subtitle = (f"{period}，{n_users:,} 人；离段质心不超过半径的连续记录合为一段，时长不少于最短时长计为停留\n"
            f"人均停留点 = 停留段数 ÷ 总人数；“差” = 同一行里最高值比最低值多多少；只测了 {len(radii)}×{len(mins)} 个组合")


viz.finish(fig, title, subtitle, SOURCE, "D06_04_停留参数敏感性.png")
plt.show()

结论：半径从 300 米到 1 公里，人均停留点只差 0.2 个左右——对半径不敏感，选常用的 500 米即可。最短时长从 15 分钟到 60 分钟，人均停留点从 4.1 个降到 2.7 个——对时长敏感，所以“停留至少 30 分钟”必须写进报告的口径说明。

## 业务参数整理

采用yaml文件对业务参数进行管理：

1. 第一步创建：configs/settings.yaml
```yaml
# config/settings.yaml —— 业务参数的唯一来源（AGENTS.md 第 8 条）
# 改参数改这里，不改代码；代码里不写 500 米、30 分钟这类业务数字。
# 一律 UTF-8 保存（Windows 记事本另存为时选 UTF-8）。
version: "1.0"

hotspot:                 # 热点：D04 定的，D06 起从这里读
  name: 西塔
  top_n: 5               # 连接用户数前几的站点参与加权，定热点中心
  radius_m: 500          # 热点范围：离中心多少米以内算“在西塔”

stay:                    # 停留识别（D06）
  radius_m: 500          # 按距离切段的半径：吸收乒乓切换；300–1000 米结果差别不大（D06 敏感性分析）
  min_minutes: 30        # 停留的最短时长：结果对它敏感，必须写进口径
```
2. 修改pyproject.toml
```toml
dependencies = [
    "pandas>=3.0.6",
    "numpy>=2.5.3",
    "matplotlib>=3.11.2",
    "pyyaml>=6.0.1"  # 用这个读取setting配置文件,这里是改的
]
```
3. 修改config问题
```python
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# @File       : config.py
# @Path       : signal-insight/src/signal_insight/config.py
# @Project    : signal-insight
# @Author     : Matri
# @Date       : 2026-10-10 14:53:18
# @Version    : v1.1.0
# @Description: 
# @ChangeLog:
#   2026-10-10 14:53:18 | Matri | v1.0.0 | 初始化创建
#   2026-10-10 16:30:00 | Matri | v1.1.0 | 新增主题配置 THEME / THEME_DIR
#   2026-10-14 09:05:00 | Matri | v1.2.0 | 主题改用viz/styles/*.mplstyle,删除THEME_DIR
import os
from pathlib import Path

import yaml


# 项目根的标记文件：根目录里只要有这个文件，就能锁定项目根，
# 不再依赖 config.py 本身所在的层级（parents 数量）。
WORKSPACE_MARKER = "signal.workspace"


def find_project_root(marker: str = WORKSPACE_MARKER) -> Path:
    """从 config.py 所在目录逐级向上查找含 marker 的目录，作为项目根路径。

    这样即使 config.py 以后移动到包内更深的层级，
    也能靠根目录里的 signal.workspace 标记文件找到正确的项目根。
    """
    for parent in Path(__file__).resolve().parents:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"未找到项目根标记文件 {marker!r}，请确认它存在于项目根目录"
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "signal.db"  # 探索库
# 检查库，前一天没做完可以从这里恢复
CHECKPOINT_DB = DATA_DIR / "checkpoints.db"

IMAGE_DIR = PROJECT_ROOT / "image"

SETTINGS_PATH = PROJECT_ROOT / "configs" / "settings.yaml"

# 默认的可视化主题: viz/style/里面的文件.default, academic. business, lively
THEME = os.environ.get("SIGNAL_THEME", "default")

# 口径常量：
UTC_OFFSET_HOURS = 8
ANALYSIS_DAY = "2018-10-03"
SOURCE_NOTE = "数据来源：沈阳移动信令样本(脱敏，2018年国庆)"
AUTHOR = "AzureKite 1210"

def load_settings(path=None) -> dict:
    p = SETTINGS_PATH if path is None else path
    try:
        text = open(p, encoding='utf-8').read()
    except UnicodeDecodeError as e:
        raise UnicodeDecodeError(e.encoding, e.object, e.start, e.end,
            f"{p} 不是UTF-8,需要更换编码重新读取"
                                )
    return  yaml.safe_load(text)
```

In [ ]:
from signal_insight import config
setting = config.load_settings()

## 驻留识别封装